In [1]:
# ── Lab 02 - Hub Release and Spaces ──
# Notes : notes/02-hub-release-and-spaces.md
# Result: results/02-release-and-spaces.md
# Goal  : release a model with versions, then create a Space
# Needs : HF_WRITE_TOKEN in the repo-root .env
# Note  : everything made here is deleted in the cleanup cell

# ── imports ──
import os, time
from pathlib import Path
from dotenv import load_dotenv
import huggingface_hub
from huggingface_hub import HfApi, hf_hub_download

# ── load the write token from .env ──
load_dotenv()
WRITE_TOKEN = os.environ["HF_WRITE_TOKEN"]

# ── start clean: no HF_TOKEN ──
os.environ.pop("HF_TOKEN", None)

# ── one API object that uses the write token ──
api = HfApi(token=WRITE_TOKEN)
USER = api.whoami()["name"]

# ── names of the things we will create ──
MODEL_REPO = f"{USER}/release-lab-model"
SPACE_REPO = f"{USER}/release-lab-space"

print("user       :", USER)
print("model repo :", MODEL_REPO)
print("space repo :", SPACE_REPO)

user       : akshayredekar74
model repo : akshayredekar74/release-lab-model
space repo : akshayredekar74/release-lab-space


In [2]:
# ── 1. create a model repo ──
# Private, so nobody else sees it. exist_ok lets you re-run the cell.

api.create_repo(MODEL_REPO, repo_type="model", private=True, exist_ok=True)
print("created: https://huggingface.co/" + MODEL_REPO)

created: https://huggingface.co/akshayredekar74/release-lab-model


In [3]:
# ── 2. upload files + model card ──
# README.md is the model card. The block between --- lines is metadata.

# ── make a local folder with two small files ──
release_dir = Path("release_files")
release_dir.mkdir(exist_ok=True)

(release_dir / "model.bin").write_bytes(b"test weights v1")
(release_dir / "README.md").write_text("""---
license: mit
language: en
tags:
  - test
---

# **Release Lab Model**

A tiny test model used to practise releases.
""")

# ── upload the whole folder in one commit ──
api.upload_folder(
    repo_id=MODEL_REPO,
    folder_path=str(release_dir),
    commit_message="Add model and model card",
)
print("files in repo:", api.list_repo_files(MODEL_REPO))

files in repo: ['.gitattributes', 'README.md', 'model.bin']


In [4]:
# ── folders: datasets/ sits next to labs/ ──
import os
from pathlib import Path

ROOT = Path.cwd().parent
DATASETS_DIR = ROOT / "datasets"
DOWNLOAD_DIR = DATASETS_DIR / "downloads"   # everything pulled from the Hub
UPLOAD_DIR = DATASETS_DIR / "upload"        # everything we push to the Hub
# for folder in (DOWNLOAD_DIR, UPLOAD_DIR):
#     folder.mkdir(parents=True, exist_ok=True)

# ── send every cache into datasets/downloads ──
# Must run BEFORE huggingface_hub / datasets / transformers are imported
os.environ["HF_HUB_CACHE"] = str(DOWNLOAD_DIR / "hub_cache")
os.environ["HF_DATASETS_CACHE"] = str(DOWNLOAD_DIR / "datasets_cache")

# # ── check: caches really moved (restart the kernel if this fails) ──
# from huggingface_hub import constants
# assert str(DOWNLOAD_DIR) in constants.HF_HUB_CACHE, "Restart the kernel, then run Cell 1 first"

In [6]:
# ── 2. upload files + model card ──
# Files live in datasets/upload/release_files

release_dir = UPLOAD_DIR / "release_files"
release_dir.mkdir(exist_ok=True)

(release_dir / "model.bin").write_bytes(b"fake weights v1")
(release_dir / "README.md").write_text("""---
license: mit
language: en
tags:
  - test
---

# Release Lab Model

A tiny fake model used to practise releases.
""")

# ── junk files we do NOT want on the Hub ──
(release_dir / "debug.pyc").write_bytes(b"junk")
(release_dir / "notes.tmp").write_text("scratch")

# ── upload the folder, skipping the junk ──
api.upload_folder(
    repo_id=MODEL_REPO,
    folder_path=str(release_dir),
    ignore_patterns=["*.pyc", "__pycache__/", "*.tmp"],
    commit_message="Add model and model card",
)
print("files in repo:", api.list_repo_files(MODEL_REPO))   # no .pyc, no .tmp

files in repo: ['.gitattributes', 'README.md', 'model.bin']


In [7]:
# ── 2a. upload the same file twice ──
# Nothing changed, so the Hub skips the commit

before = len(api.list_repo_commits(MODEL_REPO))

api.upload_file(
    path_or_fileobj=str(release_dir / "README.md"),
    path_in_repo="README.md",
    repo_id=MODEL_REPO,
)

after = len(api.list_repo_commits(MODEL_REPO))
print("commits before:", before)
print("commits after :", after)      # same number

No files have been modified since last commit. Skipping to prevent empty commit.


commits before: 9
commits after : 9


In [ ]:
# ── 3. release v1.0 ──
# A tag is a name for one exact commit. It never moves.

api.create_tag(
    MODEL_REPO,
    tag="v1.0",
    tag_message="First release",
)
print("tag v1.0 created")

In [ ]:
# ── 4. change the model, release v1.1 ──

# ── upload a new version of model.bin ──
api.upload_file(
    path_or_fileobj=b"new test weights v2 (better)", # The actual content to upload. 
    # It can be a local file path, raw bytes, or a file-like object.
    path_in_repo="model.bin", # Adds or overwrites model.bin
    repo_id=MODEL_REPO,
    commit_message="Improve weights",
)

# ── tag the new commit ──
api.create_tag(MODEL_REPO, tag="v1.1", tag_message="Better weights")

# ── list all tags ──
refs = api.list_repo_refs(MODEL_REPO)
print("tags:", [t.name for t in refs.tags])

In [10]:
# ── 5. download a specific version ──
# revision= can be a tag, a branch, or a commit id

old_path = hf_hub_download(MODEL_REPO, "model.bin", revision="v1.0", token=WRITE_TOKEN)
new_path = hf_hub_download(MODEL_REPO, "model.bin", revision="v1.1", token=WRITE_TOKEN)

print("v1.0 contains:", Path(old_path).read_bytes())
print("v1.1 contains:", Path(new_path).read_bytes())

model.bin: reconstructing file:   0%|          |  0.00B /  15.0B            

model.bin: downloading bytes:           |  0.00B            

model.bin: reconstructing file:   0%|          |  0.00B /  28.0B            

model.bin: downloading bytes:           |  0.00B            

v1.0 contains: b'test weights v1'
v1.1 contains: b'new test weights v2 (better)'


In [11]:
# ── 6. work on a branch ──
# A branch is a separate line of work. main stays untouched.

# ── create the branch ──
api.create_branch(MODEL_REPO, branch="dev", exist_ok=True)

# ── upload an experiment to dev only ──
api.upload_file(
    path_or_fileobj=b"experimental notes",
    path_in_repo="experiment.txt",
    repo_id=MODEL_REPO,
    revision="dev",
    commit_message="Add experiment",
)

# ── compare the two branches ──
print("main files:", api.list_repo_files(MODEL_REPO, revision="main"))
print("dev files :", api.list_repo_files(MODEL_REPO, revision="dev"))

No files have been modified since last commit. Skipping to prevent empty commit.


main files: ['.gitattributes', 'README.md', 'model.bin']
dev files : ['.gitattributes', 'README.md', 'experiment.txt', 'model.bin']


In [12]:
# ── 7. commit history ──
# Every upload above made one commit

for c in api.list_repo_commits(MODEL_REPO, revision="main"):
    print(c.commit_id[:8], "|", c.title)

d94154fc | Add model and model card
50c808fa | Add model and model card
8ee7f148 | Add model and model card
a921340b | Add model and model card
ac926725 | Improve weights
5c21511d | Add model and model card
1427e90a | Improve weights
3b71e608 | Add model and model card
0a744997 | initial commit


In [13]:
# ! uv pip install -qU --force-reinstall huggingface_hub gradio

In [14]:
import huggingface_hub
print(huggingface_hub.__file__)
print(huggingface_hub.__version__)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/__init__.py
2.5.0


In [ ]:
# import gradio as gr

# demo = gr.Interface(
#     fn=lambda text: text,
#     inputs="text",
#     outputs="text",
# )

# demo.launch(share=True)

In [5]:
api.create_repo(
    SPACE_REPO,
    repo_type="space",
    space_sdk="static",
    private=True,
    exist_ok=True,
)
print("created: https://huggingface.co/spaces/" + SPACE_REPO)

created: https://huggingface.co/spaces/akshayredekar74/release-lab-space


In [6]:
api.upload_file(
    path_or_fileobj="static-space/README.md",
    path_in_repo="README.md",
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Add static Space README",
)

api.upload_file(
    path_or_fileobj="static-space/index.html",
    path_in_repo="index.html",
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Add echo web page",
)

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/spaces/akshayredekar74/release-lab-space/commit/222c4d897d104eb31f2aa2c95a6e82b43e48e33e', commit_message='Add echo web page', commit_description='', oid='222c4d897d104eb31f2aa2c95a6e82b43e48e33e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/akshayredekar74/release-lab-space', endpoint='https://huggingface.co', repo_type='space', repo_id='akshayredekar74/release-lab-space'), pr_revision=None, pr_num=None)

In [7]:
# ── 9. upload the app ──
# A Gradio Space needs app.py. requirements.txt is for extra packages.

app_code = '''
import os
import gradio as gr

greeting = os.environ.get("GREETING", "Hello")
secret_ok = "yes" if os.environ.get("MY_SECRET") else "no"

def say_hi(name):
    return f"{greeting}, {name}! (secret is set: {secret_ok})"

gr.Interface(fn=say_hi, inputs="text", outputs="text").launch()
'''

api.upload_file(
    path_or_fileobj=app_code.encode(),
    path_in_repo="app.py",
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Add app",
)
api.upload_file(
    path_or_fileobj=b"",
    path_in_repo="requirements.txt",
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Add empty requirements",
)
print("Space files:", api.list_repo_files(SPACE_REPO, repo_type="space"))

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Space files: ['.gitattributes', 'README.md', 'app.py', 'index.html', 'requirements.txt', 'style.css']


In [8]:
# ── 10. variables and secrets ──
# Variable = visible setting. Secret = hidden value (like an API key).

api.add_space_variable(SPACE_REPO, key="GREETING", value="Welcome")
api.add_space_secret(SPACE_REPO, key="MY_SECRET", value="not-a-real-secret")

# ── check what is stored (secret values are never shown) ──
vars_now = api.get_space_variables(SPACE_REPO)
print("variables:", list(vars_now.keys()))

variables: ['GREETING']


In [9]:
# ── 11. wait for the Space ──
# Building takes 1-3 minutes. We check every 15 seconds.

for i in range(16):
    stage = api.get_space_runtime(SPACE_REPO).stage
    print(f"check {i + 1}: {stage}")
    if stage in ("RUNNING", "BUILD_ERROR", "RUNTIME_ERROR", "CONFIG_ERROR"):
        break
    time.sleep(15)

print("open it: https://huggingface.co/spaces/" + SPACE_REPO)

check 1: RUNNING
open it: https://huggingface.co/spaces/akshayredekar74/release-lab-space


In [10]:
# ── 11a. prepare a real Space folder ──
# Local folder: datasets/upload/hf-space

SPACE_DIR = UPLOAD_DIR / "hf-space"
SPACE_DIR.mkdir(exist_ok=True)

# ── README: sdk_version decides the Gradio version ──
(SPACE_DIR / "README.md").write_text('''---
title: Simple Sentiment Analyzer
colorFrom: blue
colorTo: purple
sdk: gradio
sdk_version: "5.29.0"
app_file: app.py
pinned: false
license: mit
---

# Simple Sentiment Analyzer
Analyzes text as Positive, Negative, or Neutral.
''')

# ── requirements: no gradio line here, the README controls it ──
(SPACE_DIR / "requirements.txt").write_text("# no gradio here - sdk_version in README controls it\n")

# ── the app ──
(SPACE_DIR / "app.py").write_text('''import gradio as gr

def analyze_sentiment(text):
    if not text.strip():
        return "Please enter some text."

    text_lower = text.lower()
    pos_words = ["good", "great", "happy", "love", "awesome", "excellent", "wonderful"]
    neg_words = ["bad", "hate", "sad", "terrible", "awful", "horrible", "worst"]

    p = sum(1 for w in pos_words if w in text_lower)
    n = sum(1 for w in neg_words if w in text_lower)

    if p > n:
        return f"Positive (score: +{p})"
    elif n > p:
        return f"Negative (score: -{n})"
    else:
        return "Neutral"

demo = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(label="Enter text", placeholder="Type something..."),
    outputs=gr.Textbox(label="Result"),
    title="Simple Sentiment Analyzer",
    examples=[
        ["I love this, it is great!"],
        ["This is terrible and awful."],
        ["The weather is okay today."],
    ],
)

demo.launch()
''')

print("files:", sorted(p.name for p in SPACE_DIR.iterdir()))

files: ['README.md', 'app.py', 'requirements.txt']


In [11]:
# ── 11b. deploy the whole folder ──
api.upload_folder(
    folder_path=str(SPACE_DIR),
    repo_id=SPACE_REPO,
    repo_type="space",
    commit_message="Deploy sentiment analyzer",
    ignore_patterns=["*.pyc", "__pycache__/", ".env", ".git/"],
)
print("Space files:", api.list_repo_files(SPACE_REPO, repo_type="space"))
# Now run the "wait for the Space" cell again (a deploy triggers a rebuild)

Space files: ['.gitattributes', 'README.md', 'app.py', 'index.html', 'requirements.txt', 'style.css']


In [12]:
# ── 11c. change several files in ONE commit ──
# Use this when files depend on each other (app.py + README.md).
# One commit means the Space never rebuilds from a half-updated state.
from huggingface_hub import CommitOperationAdd

new_app = (SPACE_DIR / "app.py").read_text().replace(
    'title="Simple Sentiment Analyzer"', 'title="Sentiment Analyzer v2"'
)
(SPACE_DIR / "app.py").write_text(new_app)       # keep the local copy in sync

operations = [
    CommitOperationAdd(path_in_repo="app.py", path_or_fileobj=str(SPACE_DIR / "app.py")),
    CommitOperationAdd(path_in_repo="README.md", path_or_fileobj=str(SPACE_DIR / "README.md")),
]

api.create_commit(
    repo_id=SPACE_REPO,
    repo_type="space",
    operations=operations,
    commit_message="Update app title",
)
print("committed, Space is rebuilding")

committed, Space is rebuilding


In [ ]:
#   ignore_patterns skip junk; identical uploads make no commit  -> cells 2, 2a
#   upload_folder deploys a Space; create_commit changes files together -> cells 11b, 11c

In [ ]:
# ── 12. restart the Space ──
# Needed after changing secrets or variables, if it is already running

api.restart_space(SPACE_REPO)
print("restart requested, stage:", api.get_space_runtime(SPACE_REPO).stage)